# Assignment 1: Breast Cancer Classification

Author: Tobias Beekmans  
Master ICT – Software Engineering  
DataOps Specialization Project – Individual Assignment  
Submission Date: 15.03.2026

In [43]:
from pathlib import Path

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from breast_cancer_assignment.dataset import load_data

# Load dataset
df = load_data()

# 3.0 Data Preparation

According to the CRISP-DM methodology, the data preparation phase focuses on transforming the dataset into a form suitable for modeling. [1]

Based on the findings from the data understanding phase, preparation in this project focuses on organizing the predictor and target variables, splitting the dataset into training and test subsets, and applying feature scaling where required for the selected machine learning algorithms.

## 3.1 Data Selection

All predictor variables are retained for modeling. Although the data understanding phase revealed correlations between several variables, the full predictor set is preserved in order to retain the complete diagnostic information available in the dataset.

The variable `target` represents the tumor diagnosis and is used as the response variable for the classification task.

In [44]:
# Define feature matrix and target variable
X = df.drop(columns="target")
y = df["target"]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

Feature matrix shape: (569, 30)
Target vector shape: (569,)


The resulting feature matrix contains 569 observations and 30 predictor variables, while the target vector contains 569 binary diagnostic labels.

In [45]:
print("Selected feature columns:")
print(X.columns.tolist())

Selected feature columns:
['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension']


The selected predictors include all diagnostic measurements provided in the dataset. No attributes are excluded, ensuring that the complete set of morphological features is available for model training.

## 3.2 Data Cleaning

The data understanding phase included a comprehensive data quality assessment. This assessment showed that the dataset does not contain missing values, duplicate observations, or invalid numerical measurements. Therefore, no imputation or record removal is required before modeling.

Potential outliers were identified during exploratory analysis. These observations are retained because extreme values in biomedical datasets may represent genuine biological variation rather than measurement errors. [4]

Consequently, no observations or attributes are removed during the data cleaning phase.

## 3.3 Data Construction

No additional attributes or records are constructed. The dataset already consists of domain-specific diagnostic measurements extracted from FNA images of breast masses, which were designed to support tumor classification tasks. [2]

Because these variables already encode relevant geometric and structural characteristics of tumor cells, further feature construction is not required for the modeling workflow.

## 3.4 Train–Test Split

To evaluate machine learning models, the dataset must be divided into separate training and test subsets. The training data is used to fit the machine learning models, while the test data is reserved for evaluating final model performance on previously unseen observations. This separation helps prevent overly optimistic performance estimates and allows a more realistic assessment of how well a model generalizes to new data. [3]

In classification problems it is important to preserve the original class distribution when splitting the dataset. If the proportion of malignant and benign tumors changes significantly between training and test sets, the evaluation results may become biased. For this reason, a stratified sampling strategy is used. Stratified splitting ensures that both subsets maintain approximately the same class proportions as the original dataset. [5]

The dataset is divided using an 80/20 split, where 80% of the data is used for model training and 20% is reserved for testing. This ratio is commonly used in machine learning projects because it provides sufficient training data while keeping an independent subset for evaluation. [3]

A fixed random seed (random_state = 42) is used to ensure that the data split is reproducible.

In [46]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training feature matrix:", X_train.shape)
print("Test feature matrix:", X_test.shape)

print("Training target vector:", y_train.shape)
print("Test target vector:", y_test.shape)

Training feature matrix: (455, 30)
Test feature matrix: (114, 30)
Training target vector: (455,)
Test target vector: (114,)


The output confirms that 455 observations are assigned to the training set and 114 observations to the test set, corresponding to an 80/20 split.

In [47]:
print("Training class distribution:")
print(y_train.value_counts(normalize=True))

print("Test class distribution:")
print(y_test.value_counts(normalize=True))

Training class distribution:
target
1    0.626374
0    0.373626
Name: proportion, dtype: float64
Test class distribution:
target
1    0.631579
0    0.368421
Name: proportion, dtype: float64


The class distributions of benign and malignant tumors in the training and test sets are nearly identical, confirming that stratified sampling preserved the original class balance of the dataset.

## 3.5 Feature Scaling

The predictor variables are standardized using StandardScaler. Standardization transforms each feature to have a mean of approximately 0 and a standard deviation of approximately 1, ensuring that all variables operate on a comparable numerical scale. [3]

The scaler is fitted only on the training data and then applied to both the training and test sets. This approach prevents data leakage and ensures that information from the test data does not influence the training process. [5]

In [48]:
scaler = StandardScaler()

# Fit scaler on training data
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to the test data
X_test_scaled = scaler.transform(X_test)

print("Scaled training set shape:", X_train_scaled.shape)
print("Scaled test set shape:", X_test_scaled.shape)

scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)

print(scaled_df.describe().round(2))

Scaled training set shape: (455, 30)
Scaled test set shape: (114, 30)
       mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
count       455.00        455.00          455.00     455.00           455.00   
mean         -0.00          0.00           -0.00      -0.00             0.00   
std           1.00          1.00            1.00       1.00             1.00   
min          -2.03         -2.17           -1.98      -1.47            -2.50   
25%          -0.70         -0.74           -0.70      -0.68            -0.72   
50%          -0.23         -0.10           -0.23      -0.31            -0.04   
75%           0.48          0.56            0.50       0.35             0.65   
max           4.02          4.55            4.02       5.37             3.61   

       mean compactness  mean concavity  mean concave points  mean symmetry  \
count            455.00          455.00               455.00         455.00   
mean              -0.00           -0.00            

The descriptive statistics confirm that the scaling procedure was applied correctly. Across all features, the mean values are approximately 0 and the standard deviations are approximately 1, which is the expected result of standardization.

The transformation does not change the number of observations or features, it only adjusts the numerical scale of the variables.

## 3.6 Data Integration

Data integration is not required for this project. The analysis is based on a single structured dataset in which all predictor variables and the diagnostic label are already contained within the same table.

Therefore, no merging or appending of additional data sources is necessary.

## 3.7 Data Formatting

The dataset is already available in a format compatible with classical machine learning algorithms. The predictor variables are organized in a numerical feature matrix (`X`) and the diagnosis labels in a binary target vector (`y`).

After applying the train–test split and feature scaling, the dataset structure is compatible with the machine learning workflow. [5]

## 3.8 Final Dataset for Modeling

After completing the data preparation steps, the dataset is ready for the modeling phase.

The preprocessing pipeline produced standardized training and test datasets that will serve as input for the machine learning models. The resulting data objects are:

- **X_train_scaled**: standardized training feature matrix  
- **X_test_scaled**: standardized test feature matrix  
- **y_train**: training target labels  
- **y_test**: test target labels

## 3.9 Export of Processed Data

To ensure reproducibility and allow consistent data access across notebooks, the processed datasets are exported as CSV files.

The standardized training and test feature matrices as well as the corresponding target vectors are stored in the `data/processed` directory. This separation between raw and processed data supports a transparent and reproducible workflow.

In [49]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

X_train_scaled_df.to_csv(output_dir / "X_train_scaled.csv", index=False)
X_test_scaled_df.to_csv(output_dir / "X_test_scaled.csv", index=False)

y_train.to_csv(output_dir / "y_train.csv", index=False)
y_test.to_csv(output_dir / "y_test.csv", index=False)

## References

[1] IBM Corporation (2011): *IBM SPSS Modeler CRISP-DM Guide*

[2] Street, W. N.; Wolberg, W. H.; Mangasarian, O. L. (1993): *Nuclear feature extraction for breast tumor diagnosis*

[3] James, G.; Witten, D.; Hastie, T.; Tibshirani, R. (2023): *An Introduction to Statistical Learning with Applications in Python*

[4] Sidey-Gibbons, J. A. M.; Sidey-Gibbons, C. J. (2019): *Machine learning in medicine: a practical introduction*

[5] Müller, A. C.; Guido, S. (2016): *Introduction to Machine Learning with Python*